In [489]:
import pandas as pd
import numpy as np


In [490]:
# Realizo a leitura da planilha. Uso a Engine OpenPyxl para transformar o xlsx em Dataframe
df = pd.read_excel("./Planilha todas as cidade ESAVI.xlsx", engine="openpyxl")

df.info

<bound method DataFrame.info of     Município de vacinação  Idade Evento  Data da Notificação Raça/Cor  \
0                Abadiânia          18.0  2022-02-16 00:00:00   Branca   
1                Abadiânia          18.0  2022-02-16 00:00:00    Parda   
2                Abadiânia          18.0  2022-02-16 00:00:00    Parda   
3                Abadiânia          18.0  2022-02-16 00:00:00   Branca   
4                Abadiânia          18.0  2022-02-16 00:00:00   Branca   
..                     ...           ...                  ...      ...   
447   Terezópolis de Goiás           3.0                 2021    Parda   
448   Terezópolis de Goiás           4.0                 2021    Parda   
449   Terezópolis de Goiás          40.0                 2021   Branca   
450   Terezópolis de Goiás          52.0                 2021    Parda   
451   Terezópolis de Goiás          44.0                 2021    Preta   

          Sexo                            Imunobiológico (vacina)  \
0    Mascu

In [491]:
df.describe()

,Idade Evento,Mês de gestação
count,444.000000,444.000000
mean,36.081081,0.126126
std,21.162762,1.778310
min,0.000000,0.000000
25%,18.000000,0.000000
50%,35.000000,0.000000
75%,51.000000,0.000000
max,90.000000,35.000000


In [492]:
df = df.dropna()

print (df.isna().sum()) # Aqui consigo descobrir se eliminei todos os registros NaN presentes na planilha

Município de vacinação                                0
Idade Evento                                          0
Data da Notificação                                   0
Raça/Cor                                              0
Sexo                                                  0
Imunobiológico (vacina)                               0
Dose                                                  0
Gestante                                              0
Mês de gestação                                       0
Mulher amamentando no momento da vacinação?           0
Tipo de Evento                                        0
Reação / evento adverso                               0
Classificação de gravidade                            0
Gravidade                                             0
Desfecho (evolução do caso)                           0
Doenças (CID10) - Preexistente                        0
Medicamento em uso anterior ou durante a vacinação    0
Nome do Medicamento                             

In [493]:
df['Idade Evento'] = df['Idade Evento'].astype(int)
df['Mês de gestação'] = df['Mês de gestação'].astype(int)

In [494]:
df['Dose'] = df['Dose'].str.findall(r'[1-6]') 
# Encontra os valores pelo regex (os valores encontrados de 1-6, e 'divindo' pelos números)

df['Dose'] = df['Dose'].apply(lambda x: list(dict.fromkeys(x)))
# Apply é uma função do Pandas que a gente pode utilizar para colocar uma função a ser resolvida em cada linha
# Lambda é só uma forma de definir uma função, de forma curta
# No final, ele gera um dicionário pelo número (portanto, pode apenas existir uma chave)

df_split = pd.DataFrame(df['Dose'].tolist(), index=df.index)
# To list transforma esse dicionário especificamente em uma lista ['1,2,3']

stacked = df_split.stack()
# Ele gera uma pilha de valores em lista, mantendo ordem pelo índice original

dummies = pd.get_dummies(stacked)
# Ele gera os dummies ()

dummies = dummies.groupby(level=0).max().astype(int)

dummies = dummies.rename(columns={
    '1': 'Tomou_Primeira_Dose',
    '2': 'Tomou_Segunda_Dose',
    '3': 'Tomou_Terceira_Dose',
    '4': 'Tomou_Quarta_Dose',
    '5': 'Tomou_Quinta_Dose',
    '6': 'Tomou_Sexta_Dose'
})

df = pd.concat([df, dummies], axis=1)

df = df.drop(columns='Dose')

In [495]:

df_split = (
    df['Imunobiológico (vacina)']
    .str.replace(r'^\d+:\s*', '', regex=True)
    .str.split(r'/\s*\d+:', expand=True, regex=True)
)
# Utilizo o regex para splitar em colunas as marcas
#Além disso, retiro os valores "1:/2:"

df_split = df_split.rename(columns={
    0: 'Marca_Primeira_Dose',
    1: 'Marca_Segunda_Dose',
    2: 'Marca_Terceira_Dose',
    3: 'Marca_Quarta_Dose',
    4: 'Marca_Quinta_Dose',
    5: 'Marca_Sexta_Dose'
    })
#


df_split = df_split.fillna('Não tomou')

df = pd.concat([df, df_split], axis=1)

df = df.drop(columns='Imunobiológico (vacina)')

In [496]:
df_limpo = (df['Reação / evento adverso']
        .replace(' ','')
        .str.replace(r'[\d+:\s*]', '', regex=True)
        .str.upper()
        .str.split('/',regex=True, expand=True))

stacked = df_limpo.stack()
dummies = pd.get_dummies(stacked)
dummies = dummies.groupby(level=0).max().astype(int)
dummies = dummies.add_prefix('Reação: ')

df = pd.concat([df, dummies], axis=1)

df = df.drop(columns='Reação / evento adverso')

In [497]:
df_split = (
    df['Classificação de gravidade']
    .str.replace(r'\d+:\s*', '', regex=True)  # remove "1:", "2:", ... "15:"
    .str.split('/', expand=True)
)

df_split = df_split.fillna('Sem classificação')

df = pd.concat([df, df_split], axis=1)

df = df.drop(columns='Classificação de gravidade')

df = df.rename(columns={
    0: 'Gravidade 1',
    1: 'Gravidade 2',
    2: 'Gravidade 3',
    3: 'Gravidade 4',
    4: 'Gravidade 5',
    5: 'Gravidade 6',
    6: 'Gravidade 7',
    7: 'Gravidade 8',
    8: 'Gravidade 9',
    9: 'Gravidade 10',
    10: 'Gravidade 11',
    11: 'Gravidade 12',
    12: 'Gravidade 13',
    13: 'Gravidade 14',
    14: 'Gravidade 15',
    15: 'Gravidade 16',
})


In [498]:
df_split = (
    df['Doenças (CID10) - Preexistente']
    .str.replace(r'\d+:\s*', '', regex=True)
    .str.upper()
    .str.split('/', expand=True))

df = df.drop(columns='Doenças (CID10) - Preexistente')

stacked = df_split.stack()
dummies = pd.get_dummies(stacked)
dummies = dummies.groupby(level=0).max().astype(int)
dummies = dummies.add_prefix('PréExistente: ')

df = pd.concat([df,dummies], axis=1)

In [499]:
df_split = (
    df['Desfecho (evolução do caso)']
    .str.replace(r'\d+:\s*', '', regex=True)
    .str.upper()
    .str.strip()                              
    .str.split('/', expand=True)
)

stacked = df_split.stack().str.strip()

dummies = pd.get_dummies(stacked)
dummies = dummies.groupby(level=0).max().astype(int)
dummies = dummies.add_prefix('Desfecho: ')
df = pd.concat([df, dummies], axis=1)

df = df.drop(columns=['Desfecho (evolução do caso)', 'Desfecho: NÃO', 'Desfecho: ' ])

In [500]:
stacked = (
    df['Nome do Medicamento']
    .str.replace(r'\d+:\s*', '', regex=True)
    .str.split(r'\s*/\s*')
    .explode()
    .str.replace(r'\b\d+\s*MG\b', '', regex=True)
    .str.strip()
    .str.upper()
)

dummies = pd.get_dummies(stacked).groupby(level=0).max().astype(int)
dummies = dummies.add_prefix('Medicamentos Tomados: ')

df = pd.concat([df, dummies], axis=1)

df = df.drop(columns=['Nome do Medicamento', 'Medicamentos Tomados: NÃO'])

In [501]:
df_split = (
    df['Tipo de Evento']
    .str.replace(r'\d+:\s*', '', regex=True)
    .str.upper()
    .str.strip()
    .str.split(' / ', expand=True)
)

stacked = df_split.stack()

dummies = pd.get_dummies(stacked)
dummies = dummies.groupby(level=0).max().astype(int)
dummies = dummies.add_prefix('Tipo de Evento: ')

df = pd.concat([df, dummies], axis=1)

df = df.drop(columns='Tipo de Evento')

In [502]:
df_split = (
    df['Tipo de Atendimento']
    .str.replace(r'\d+:\s*', '', regex=True)
    .str.upper()
    .str.strip()
    .str.split(' / ', expand=True)
)

stacked = df_split.stack()

dummies = pd.get_dummies(stacked)
dummies = dummies.groupby(level=0).max().astype(int)
dummies = dummies.add_prefix('Tipo Atendimento: ')

df = pd.concat([df, dummies], axis=1)

df = df.drop(columns='Tipo de Atendimento')

In [503]:
df.to_csv('teste.csv')

# Explicação do Código — `split`, `stack`, `dummies` e Transformações no Pandas

Seu código realiza um processo de **limpeza**, **normalização** e **transformação categórica** dos dados da planilha ESAVI utilizando principalmente:

- `split`
- `stack`
- `get_dummies`
- `groupby`
- `concat`

A ideia principal é transformar colunas textuais complexas em colunas estruturadas e utilizáveis para análise estatística ou Machine Learning.

---

# 1. O que é `split`?

O `split` serve para **dividir strings** em várias partes.

Exemplo simples:

```python
texto = "Febre/Dor de cabeça/Náusea"

texto.split("/")

# 2. Split “Puro” vs Dummies

- Split puro

O split puro apenas separa os dados em colunas.

- Dummies

Dummies transformam categorias em valores binários (0 e 1).